In [1]:
import os
os.chdir('./stat_csv')
os.getcwd()

'/home/tako/Kasetsart/statistics/stat_csv'

In [2]:
#! pip install gensim

In [3]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

### Load and Split Data

In [4]:
data = pd.read_csv('yelp_labelled_edited.txt', sep="\t")
data.head()

,Text,Review
0,Wow... Loved this place.,1
1,Crust is not good.,0
2,Not tasty and the texture was just nasty.,0
3,Stopped by during the late May bank holiday of...,1
4,The selection on the menu was great and so wer...,1


In [5]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(data['Text'], data['Review'], test_size=0.2, random_state=42)

In [6]:
print(X_train.shape)
print(X_test.shape)

(800,)
(200,)


### Text Preprocessing

In [7]:
import nltk
custom_dir = '.devenv/state/venv/nltk_data' 
nltk.download('punkt_tab', download_dir=custom_dir)
nltk.download('stopwords', download_dir=custom_dir)
nltk.download('wordnet', download_dir=custom_dir)

[nltk_data] Downloading package punkt_tab to
[nltk_data]     .devenv/state/venv/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     .devenv/state/venv/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     .devenv/state/venv/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [8]:
def preprocess_text(text):
    import re, string
    from nltk.tokenize import word_tokenize 
    from nltk.corpus import stopwords 
    from nltk.stem import WordNetLemmatizer    
    
    word_tokens = word_tokenize(text)
    word_tokens = [word.lower() for word in word_tokens] 
    word_tokens = [re.sub(r'[^\w\s]', '', token) for token in word_tokens if re.sub(r'[^\w\s]', '', token)]
    corpus_stop_words = set(stopwords.words('english')) 
    word_tokens = [word for word in word_tokens if not word in corpus_stop_words] 
    lemmatizer = WordNetLemmatizer()
    word_tokens = [lemmatizer.lemmatize(word) for word in word_tokens]
    return word_tokens

In [9]:
processed_X_train = [preprocess_text(word) for word in X_train]
processed_X_train

[['worst', 'salmon', 'sashimi'],
 ['excellent', 'new', 'restaurant', 'experienced', 'frenchman'],
 ['went', 'lunch', 'service', 'slow'],
 ['think', 'restaurant', 'suffers', 'trying', 'hard', 'enough'],
 ['lunch', 'great', 'experience'],
 ['got', 'home', 'see', 'driest', 'damn', 'wing', 'ever'],
 ['delicious'],
 ['brought', 'fresh', 'batch', 'fry', 'thinking', 'yay', 'something', 'warm'],
 ['tragedy', 'struck'],
 ['awful', 'service'],
 ['great',
  'food',
  'service',
  'huge',
  'portion',
  'give',
  'military',
  'discount'],
 ['waiter', 'nt', 'helpful', 'friendly', 'rarely', 'checked', 'u'],
 ['could', 'barely', 'stomach', 'meal', 'nt', 'complain', 'business', 'lunch'],
 ['price', 'think', 'place', 'would', 'much', 'rather', 'gone'],
 ['fried', 'rice', 'dry', 'well'],
 ['truly', 'unbelievably', 'good', 'glad', 'went', 'back'],
 ['eclectic', 'selection'],
 ['based',
  'subpar',
  'service',
  'received',
  'effort',
  'show',
  'gratitude',
  'business',
  'wo',
  'nt',
  'going',
  

In [10]:
processed_X_test = [preprocess_text(word) for word in X_test]
processed_X_test

[['nt', 'gone', 'go'],
 ['try',
  'airport',
  'experience',
  'tasty',
  'food',
  'speedy',
  'friendly',
  'service'],
 ['restaurant', 'clean', 'family', 'restaurant', 'feel'],
 ['personally',
  'love',
  'hummus',
  'pita',
  'baklava',
  'falafel',
  'baba',
  'ganoush',
  'amazing',
  'eggplant'],
 ['come', 'hungry', 'leave', 'happy', 'stuffed'],
 ['great', 'place', 'highly', 'recommend'],
 ['best',
  'luck',
  'rude',
  'noncustomer',
  'service',
  'focused',
  'new',
  'management'],
 ['reasonably', 'priced', 'also'],
 ['worst', 'foodservice'],
 ['seriously', 'solid', 'breakfast'],
 ['service', 'terrible', 'though'],
 ['2', 'time', 'bad', 'customer', 'service'],
 ['tried', 'go', 'lunch', 'madhouse'],
 ['food', 'nt', 'good'],
 ['meat', 'pretty', 'dry', 'sliced', 'brisket', 'pulled', 'pork'],
 ['overall', 'great', 'experience'],
 ['went', 'bachi', 'burger', 'friend', 'recommendation', 'disappointed'],
 ['ordered',
  'albondigas',
  'soup',
  'warm',
  'tasted',
  'like',
  'toma

###### Expected processed_train output = Nested list
###### sublist = list of tokenized words from each review
###### [['greek', 'dressing', 'creamy', 'flavorful'],
######  ['food', 'amazing'],
######  ['never', 'going', 'back'],
######  ['loved', 'place'],
######  ['came', 'back', 'today', 'since', 'relocated', 'still', 'impressed'],
######  ['liked', 'patio', 'service', 'outstanding'],..]


### Text Representation

In [11]:
# Convert nested list to word2vec vectors

In [12]:
selected_vector_size = 50

from gensim.models import Word2Vec
w2v_model = Word2Vec(sentences=processed_X_train,
                 vector_size=selected_vector_size,   # size of word embeddings
                 window=5,          # context window size
                 min_count=1,       # include all words
                 workers=4,         # number of CPU threads used to train the model in parallel.
                 sg=0               # CBOW
                )

In [13]:
def average_vectors(token_vector, vector_size=100):
    # token_vector = list of tokens or vocabs from one review
    # This function will:
    # 1) retrieve word vectors of all vocabs from models
    # 2) average word vectors to get one representative vector for each review

    # Get word vectors from CBOW model
    word_vectors = [w2v_model.wv[word] for word in token_vector if word in w2v_model.wv]
    # If no word exists in the model, return array of 100 zeros
    if len(word_vectors) == 0:
        return np.zeros(vector_size)
    # Convert word vectors to numpy arrays
    word_vectors_np = np.array(word_vectors)
    # Average numerical word vectors and return
    return word_vectors_np.mean(axis=0)

X_train_vectors = np.array([average_vectors(token_vector, selected_vector_size) for token_vector in processed_X_train])
X_test_vectors = np.array([average_vectors(token_vector, selected_vector_size) for token_vector in processed_X_test])

### Training Model and Evaluate Performance

In [14]:
# Classification model using X_train_vectors nd y_train as training dataset
# Use X_test_vectors as test dataset

In [15]:
from sklearn.linear_model import LogisticRegression
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_vectors, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [16]:
from sklearn.tree import DecisionTreeClassifier
clf = DecisionTreeClassifier(criterion = 'gini', random_state = 0)
clf.fit(X_train_vectors, y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [17]:
selected_n = 100  # n_estimators  = number of decision trees used in random forest model, default value = 100
from sklearn.ensemble import RandomForestClassifier
clf = RandomForestClassifier(n_estimators = selected_n, criterion = 'gini', random_state = 0)
clf.fit(X_train_vectors, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [18]:
y_predicted = clf.predict(X_test_vectors)
# Evalute performance
from sklearn.metrics import classification_report
print(classification_report(y_test,y_predicted))

              precision    recall  f1-score   support

           0       0.65      0.69      0.67        96
           1       0.69      0.65      0.67       104

    accuracy                           0.67       200
   macro avg       0.67      0.67      0.67       200
weighted avg       0.67      0.67      0.67       200



In [19]:
print(y_predicted)

[0 1 0 1 1 1 0 1 0 0 0 1 1 0 0 1 0 0 1 1 1 1 0 0 0 0 1 1 1 0 1 1 1 1 0 1 1
 0 1 0 0 0 1 1 1 0 0 0 1 0 0 1 0 1 0 1 1 0 0 1 1 0 1 1 0 0 0 1 0 0 1 1 1 0
 1 1 0 0 0 0 1 1 0 1 0 1 0 1 1 1 1 0 1 1 0 0 1 0 0 1 0 0 1 1 1 1 1 0 0 0 0
 0 1 0 0 1 1 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 0 1 1 0 1 1 0 0 1 0 0 0 0 0 0
 0 0 0 1 0 0 0 0 1 0 0 0 1 1 0 0 0 1 0 1 1 1 1 1 1 0 0 0 1 0 1 0 0 0 1 0 1
 1 1 0 0 0 1 1 1 0 0 1 1 1 0 1]


In [20]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
print('Accuracy:', accuracy_score(y_test, y_predicted))
print('Precision:', precision_score(y_test, y_predicted, pos_label=1))
print('Recall:', recall_score(y_test, y_predicted, pos_label=1))
print('F1 score:', f1_score(y_test, y_predicted, pos_label=1))

Accuracy: 0.67
Precision: 0.6938775510204082
Recall: 0.6538461538461539
F1 score: 0.6732673267326733


In [21]:
from sklearn.metrics import confusion_matrix
print(confusion_matrix(y_test, y_predicted))

[[66 30]
 [36 68]]


In [22]:
### Try more classification models here